In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn
%matplotlib inline

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [3]:
df=pd.read_csv('../data/healthcare_cleaned.csv')

In [4]:
drop_cols=['patient_id','treatment_cost','mortality_risk','diagnosis_category','length_of_stay','readmission_30day']

In [5]:
X=df.drop(drop_cols,axis=1)
y=df['treatment_cost']

In [6]:
y_log=np.log1p(y)

In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y_log,test_size=0.2,random_state=42)

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
cat_cols=[col for col in X.columns if X[col].dtypes=='str']
num_cols=[col for col in X.columns if col not in cat_cols]
preprocessor=ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(),cat_cols)
])
linear_reg_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
])
linear_reg_pipeline.fit(X_train,y_train)
y_pred=linear_reg_pipeline.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))
print(f" the real variance check")

0.3557148230206698
0.2001841226880628
 the real variance check


In [17]:
from sklearn.ensemble import RandomForestRegressor
random_forest_reg_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor())
])
random_forest_reg_pipeline.fit(X_train,y_train)
y_pred=random_forest_reg_pipeline.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))

0.31830482648750413
0.20591390218704408


In [18]:
from sklearn.model_selection import GridSearchCV
params={
    'model__n_estimators':[50,100,150],
    'model__max_depth':[3,5,7],
    'model__min_samples_split':[3,5,8,10]
}
grid_model_rf=GridSearchCV(param_grid=params,estimator=random_forest_reg_pipeline,n_jobs=-1,cv=5,scoring='r2')
grid_model_rf.fit(X_train,y_train)
y_pred=grid_model_rf.predict(X_test)
print(r2_score(y_test,y_pred))

0.3513110739799765


In [19]:
print(grid_model_rf.best_params_)
print(grid_model_rf.best_score_)

{'model__max_depth': 7, 'model__min_samples_split': 3, 'model__n_estimators': 150}
0.3578696220308024


In [20]:
from xgboost import XGBRegressor
XGB_reg_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',XGBRegressor())
])
XGB_reg_pipeline.fit(X_train,y_train)
y_pred=XGB_reg_pipeline.predict(X_test)
print(r2_score(y_test,y_pred))
print(root_mean_squared_error(y_test,y_pred))

0.31797661200344174
0.2059634667100706


In [21]:
from sklearn.model_selection import GridSearchCV
params={
    'model__n_estimators':[50,100,150],
    'model__max_depth':[3,5,7],
    'model__learning_rate':[0.01,0.3,0.1]
}
grid_model_xgb=GridSearchCV(param_grid=params,estimator=XGB_reg_pipeline,n_jobs=-1,cv=5,scoring='r2')
grid_model_xgb.fit(X_train,y_train)
y_pred=grid_model_xgb.predict(X_test)
print(r2_score(y_test,y_pred))

0.357567356286672


In [22]:
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)

{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
0.3620857790042785


In [23]:
y_pred_final = np.expm1(grid_model_xgb.predict(X_test))
y_test_original = np.expm1(y_test)
print(root_mean_squared_error(y_test_original, y_pred_final))

8033.389946069594


1) XGBoost best model R²=0.358, RMSE=$8,033 on original scale
2) Log transform applied — skew=1.16 confirmed it necessary
3) All models hit same ceiling ~0.35 — weak feature signal
4) Linear Regression competitive with tree models — 
   suggests relationships are mostly linear for treatment cost

In [ ]:
import joblib
joblib.dump(grid_model_xgb, '../models/treatment_cost_model.pkl')

FileNotFoundError: [Errno 2] No such file or directory: '../models/treatment_cost_model.pkl'